In [20]:
import torch
from torch import nn
import numpy as np
import pandas as pd
import dataclasses
from dataclasses import dataclass, field
from typing import List
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from template_modules import EncoderStaticBase, EncoderStaticBaseConfig
from collections.abc import Iterable
from basic_conv1d import bn_drop_lin
import warnings
warnings.filterwarnings('ignore')
from clinical_ts.template_modules import ShapeConfig

In [21]:
# ----------------------------------------
# Config (matches YAML)
# ----------------------------------------
BATCH_SIZE = 32
EPOCHS     = 40
LR         = 0.001
WEIGHT_DECAY = 0.001
DROPOUT    = 0.5
LIN_FTRS   = [128, 128, 128]

In [22]:
# ----------------------------------------
# 1. Load data (same as multimodal.py)
# ----------------------------------------
print("Loading data...")
df = pd.read_csv('/fs/dss/home/gaad2403/MDS-ED/src/data/memmap/mds_ed.csv',
                 low_memory=False)
print(f"shape: {df.shape}")

Loading data...


FileNotFoundError: [Errno 2] No such file or directory: '/fs/dss/home/gaad2403/MDS-ED/src/data/memmap/mds_ed.csv'

In [ ]:
input_cols = [c for c in df.columns if c.split("_")[0] in ['biometrics','demographics','labvalues','vitals']]

# median imputation based on training set
df_train = df[df['general_strat_fold'] < 18]
train_medians = df_train[input_cols].median().to_dict()
train_nans = [c for c,v in df_train[input_cols].isna().sum().to_dict().items() if v > 0]
for c in train_nans:
    df.loc[df[c].isna(), c] = train_medians[c]
df = df.copy()

In [ ]:
# categorical vs continuous feature split (same as multimodal.py)
unique_counts = {c: len(np.unique(np.array(df[c]))) for c in input_cols}
cat_features = [c for c,v in unique_counts.items() if v < 10 and not c.endswith("nan") and not c.startswith("labvalues")]
cat_features_dim = [unique_counts[c] for c in cat_features]
cont_features = [c for c in input_cols if c not in cat_features]

print(f"categorical features: {len(cat_features)} dimensions: {cat_features_dim} continuous features: {len(cont_features)}")

In [ ]:
# vitals_acuity: convert from 1-based to 0-based indexing
df["vitals_acuity"] = df["vitals_acuity"].apply(lambda x: int(x)-1)

# ethnicity → categorical encoding (same as multimodal.py)
lbl_itos_ethnicity = [
    'demographics_ethnicity_asian',
    'demographics_ethnicity_black/african',
    'demographics_ethnicity_hispanic/latino',
    'demographics_ethnicity_other',
    'demographics_ethnicity_white'
]

df["demographics_ethnicity"] = df.apply(
    lambda row: np.where([row[c] for c in lbl_itos_ethnicity])[0][0], axis=1
)
df.drop(lbl_itos_ethnicity, axis=1, inplace=True)

In [ ]:
input_cols = [c for c in df.columns if c.split("_")[0] in ['biometrics','demographics','labvalues','vitals']]
cat_features = [c for c in cat_features if c in df.columns]
cat_features = [c for c in input_cols if c in cat_features]
cont_features = [c for c in input_cols if c not in cat_features]

# deterioration labels
lbl_itos_deterioration = ["mortality_1d","icu_24h","cardiac_arrest","vasopressors"]
for c in lbl_itos_deterioration:
    df["deterioration_"+c] = df["deterioration_"+c].replace(-999., np.nan)

In [ ]:
target_cols = [
    "deterioration_mortality_1d",
    "deterioration_icu_24h",
    "deterioration_icu_24h",
    "deterioration_vasopressors"
]

In [ ]:
# 3. Train / Validation / Test split
# ----------------------------------------
train_df = df[df['general_strat_fold'].isin(range(0, 18))].reset_index(drop=True)
val_df   = df[df['general_strat_fold'] == 18].reset_index(drop=True)
test_df  = df[df['general_strat_fold'] == 19].reset_index(drop=True)
# only first ECG per hospital stay for validation/test
val_df   = val_df[val_df['general_ecg_no_within_stay'] == 0].reset_index(drop=True)
test_df  = test_df[test_df['general_ecg_no_within_stay'] == 0].reset_index(drop=True)

print(f"Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# 4. Dataset
# ----------------------------------------
class TabularDataset(Dataset):
    def __init__(self, df, cont_features, cat_features, lbl_cols):
        self.cont = torch.tensor(df[cont_features].values, dtype=torch.float32)
        self.cat  = torch.tensor(df[cat_features].values, dtype=torch.long)
        self.labels = torch.tensor(df[["deterioration_"+c for c in lbl_cols]].values, dtype=torch.float32)

    def __len__(self):
        return len(self.cont)

    def __getitem__(self, idx):
        return self.cont[idx], self.cat[idx], self.labels[idx]

train_ds = TabularDataset(train_df, cont_features, cat_features, lbl_itos_deterioration)
val_ds   = TabularDataset(val_df,   cont_features, cat_features, lbl_itos_deterioration)
test_ds  = TabularDataset(test_df,  cont_features, cat_features, lbl_itos_deterioration)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
# 5. Model initialization (BasicEncoderStaticMLP unchanged)
# ----------------------------------------
@dataclass
class MLPConfig:
    embedding_dims: List[int] = field(default_factory=lambda: [16, 16, 16])
    vocab_sizes: List[int]    = field(default_factory=lambda: [2, 5, 5])
    lin_ftrs: List[int]       = field(default_factory=lambda: [128, 128, 128])
    dropout: float = 0.5
    batch_norm: bool = True

@dataclass
class ShapeCfg:
    static_dim: int = 0
    static_dim_cat: int = 0
    channels: int = 0
    length: int = 0
    sequence_last: bool = False
    channels2: int = 0

shape = ShapeCfg(static_dim=len(cont_features), static_dim_cat=len(cat_features))

mlp_cfg = MLPConfig(
    embedding_dims=[unique_counts[c] for c in cat_features],
    vocab_sizes=[unique_counts[c] for c in cat_features],
    lin_ftrs=LIN_FTRS
)

encoder = BasicEncoderStaticMLP(mlp_cfg, shape, target_dim=len(lbl_itos_deterioration))
print(f"\nModel parameter count: {sum(p.numel() for p in encoder.parameters()):,}")